# Baseline category and subcategory classification

Phi-4-mini, architectures A (event and context) and C (event, context and relations); training seed 42.


In [ ]:
import os, sys, importlib.util
print('Python:', sys.version)
print('Kernel:', sys.executable)
print('SLURM_JOB_ID:', os.environ.get('SLURM_JOB_ID'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('PYTHONPATH:', os.environ.get('PYTHONPATH'))
required = ['torch','transformers','datasets','peft','trl','accelerate','bitsandbytes']
for name in required:
    print(f'{name:14}:', 'available' if importlib.util.find_spec(name) else 'missing')


## Dependencies


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This kernel has no CUDA GPU. Use a GPU kernel for fine tuning.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))


In [ ]:
from pathlib import Path
import json, csv, gc
from collections import Counter
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SPLITS = ROOT / 'models/phi4_mini/classification/baseline/splits'
OUTPUTS = ROOT / 'models/phi4_mini/classification/baseline/adapters'
RESULTS = ROOT / 'results/phi4_mini/classification/baseline'
OUTPUTS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
for f in ('train.jsonl','val.jsonl','test.jsonl','taxonomy.json'):
    assert (SPLITS / f).exists(), f'Missing split {f}; run 01_prepare.ipynb first'

MODEL_ID = 'microsoft/Phi-4-mini-instruct'
SEED = 42
RELATIONS = ['xIntent','xNeed','xAttr','xEffect','xReact','xWant',
             'oEffect','oReact','oWant','isAfter','HasSubEvent','isBefore',
             'HinderedBy','Causes','xReason','isFilledBy']

MAX_ITEMS_PER_RELATION = 0

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return [json.loads(s) for s in f if s.strip()]

train_rows = read_jsonl(SPLITS/'train.jsonl')
val_rows = read_jsonl(SPLITS/'val.jsonl')
test_rows = read_jsonl(SPLITS/'test.jsonl')
taxonomy = json.loads((SPLITS/'taxonomy.json').read_text(encoding='utf-8'))
print('Train / Val / Test:', len(train_rows), len(val_rows), len(test_rows))
print('Categories:', list(taxonomy))


In [ ]:
import hashlib

for name in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    path = SPLITS / name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    print(name, digest)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TAXONOMY_TEXT = '\n'.join(f'- {cat}: {", ".join(sub)}' for cat, sub in sorted(taxonomy.items()))

def make_prompt(record, architecture):
    assert architecture in ('A', 'C')
    system = f"""
You are a forensic analysis expert.

Your task is to classify each provided forensic
scenario by assigning exactly one crime category
and one corresponding crime subcategory.

### Classification Taxonomy ###

{TAXONOMY_TEXT}

### Classification Rules ###

1. Analyze the event and all the information
   explicitly provided for the scenario.

2. Assign exactly one crime category and
   one corresponding crime subcategory.

3. Use only the categories and subcategories
   listed in the taxonomy. Do not create
   additional labels.

4. If multiple classifications could apply,
   select the one that best represents
   the main criminal conduct.

5. Do not invent facts, intentions or
   circumstances unsupported by the
   provided information.

6. Treat any stated intentions, consequences
   or inferred relations as contextual evidence,
   not necessarily as established facts.

### Output Format ###

Return ONLY a valid JSON object:

{{
  "crime_category": "category",
  "crime_subcategory": "subcategory"
}}

Do not provide additional explanations.
Do not use markdown.
"""
    user = f'Event: {record["event"]}\nBrief context: {record["brief_context"]}'
    if architecture == 'C':
        rel_lines = []
        for rel in RELATIONS:
            values = record[rel]
            if MAX_ITEMS_PER_RELATION:
                values = values[:MAX_ITEMS_PER_RELATION]
            rel_lines.append(f'{rel}: {json.dumps(values, ensure_ascii=False)}')
        user += '\nATOMIC forensic relation annotations:\n' + '\n'.join(rel_lines)
    return tokenizer.apply_chat_template(
        [{'role':'system','content':system}, {'role':'user','content':user}],
        tokenize=False, add_generation_prompt=True
    )

def target_text(r):
    return json.dumps({'crime_category': r['crime_category'],
                       'crime_subcategory': r['crime_subcategory']},
                      ensure_ascii=False, separators=(',', ':')) + '<|end|>'

print('Example prompt A:\n', make_prompt(train_rows[0], 'A')[:850])
print('Example prompt C (first 1100 characters):\n', make_prompt(train_rows[0], 'C')[:1100])


## Check sequence lengths


In [ ]:
def check_lengths(architecture, train_limit=0, val_limit=0, max_length=None):
    if max_length is None:
        max_length = 3072 if architecture == 'A' else 4096
    for name, rows, limit in [('train', train_rows, train_limit), ('val', val_rows, val_limit)]:
        subset = rows[:limit] if limit else rows
        lengths = [len(tokenizer(make_prompt(r,architecture) + target_text(r),
                                 add_special_tokens=False)['input_ids']) for r in subset]
        if not lengths:
            raise ValueError(f'{name}: empty split')
        print(architecture, name, 'n:',len(lengths), 'max:',max(lengths),
              'oltre',max_length,':',sum(x>max_length for x in lengths))
        if max(lengths) > max_length:
            raise ValueError(f'{architecture}: max={max(lengths)} oltre {max_length}. '
                             'Aumenta max_length se VRAM sufficiente, oppure riduci MAX_ITEMS_PER_RELATION.')
    return max_length

check_lengths('A', train_limit=64, val_limit=16)
check_lengths('C', train_limit=64, val_limit=16)


## QLoRA training


In [ ]:
def train_arch(architecture, *, full=False, epochs=None, max_length=None):
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU unavailable')
    assert architecture in ('A','C')
    max_length = max_length or (3072 if architecture == 'A' else 4096)
    limit_train = 0 if full else 64
    limit_val = 0 if full else 16
    epochs = (2 if full else 1) if epochs is None else epochs
    check_lengths(architecture, limit_train, limit_val, max_length)
    name = architecture if full else architecture+'_smoke'
    target_dir = OUTPUTS / name

    src_train = train_rows[:limit_train] if limit_train else train_rows
    src_val = val_rows[:limit_val] if limit_val else val_rows
    def convert(r):
        return {'prompt': make_prompt(r, architecture), 'completion': target_text(r)}
    ds_train = Dataset.from_list([convert(r) for r in src_train])
    ds_val = Dataset.from_list([convert(r) for r in src_val])

    use_bf16 = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    print('Loading base model...', MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quant, torch_dtype=dtype,
        trust_remote_code=True, device_map={'':torch.cuda.current_device()}
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(task_type='CAUSAL_LM', r=16, lora_alpha=32,
                      lora_dropout=0.05, target_modules='all-linear')
    config = SFTConfig(
        output_dir=str(target_dir), max_length=max_length, completion_only_loss=True,
        num_train_epochs=epochs, per_device_train_batch_size=1,
        per_device_eval_batch_size=1, gradient_accumulation_steps=8,
        learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
        logging_steps=10, logging_strategy='steps',logging_first_step=True, save_strategy='epoch', eval_strategy='epoch',
        save_total_limit=2, bf16=use_bf16, fp16=not use_bf16,
        gradient_checkpointing=True, report_to='none', packing=False, seed=SEED
    )
    trainer = SFTTrainer(model=model, args=config, train_dataset=ds_train,
                         eval_dataset=ds_val, processing_class=tokenizer,
                         peft_config=lora)
    try:
        trainer.train()
        trainer.save_model(str(target_dir))
        tokenizer.save_pretrained(str(target_dir))
        (target_dir/'experiment.json').write_text(json.dumps({
            'architecture':architecture, 'model':MODEL_ID, 'full':full,
            'epochs':epochs, 'max_length':max_length,
            'max_items_per_relation':MAX_ITEMS_PER_RELATION,
            'seed':SEED
        }, indent=2), encoding='utf-8')
        print('Adapter saved in:', target_dir)
    finally:
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
    return str(target_dir)


## Optional smoke runs


## Full training


In [ ]:
adapter_C = train_arch('C', full=True)


## Base and adapter predictions


In [ ]:
def parse_answer(raw):
    s = raw.replace('```json','').replace('```','').replace('<|end|>','').strip()
    try:
        start = s.index('{')
        obj, _ = json.JSONDecoder().raw_decode(s[start:])
    except (ValueError, json.JSONDecodeError):
        return None, False, False
    if not isinstance(obj, dict):
        return None, True, False
    c,s = obj.get('crime_category'),obj.get('crime_subcategory')
    valid_pair = isinstance(c, str) and isinstance(s, str) and s in taxonomy.get(c, [])
    return obj, True, valid_pair

@torch.inference_mode()
def predict_arch(architecture, *, adapter=None, name=None, limit=30, max_new_tokens=128):
    assert architecture in ('A','C')
    if adapter is not None:
        config_file = Path(adapter)/'experiment.json'
        if not config_file.exists():
            raise FileNotFoundError(f'Missing {config_file}: training incomplete?')
        cfg = json.loads(config_file.read_text(encoding='utf-8'))
        if cfg['architecture'] != architecture or cfg['max_items_per_relation'] != MAX_ITEMS_PER_RELATION:
            raise ValueError('Inference prompt must match training')
    name = name or (architecture + ('_ft' if adapter else '_base'))
    chosen = test_rows[:limit] if limit else test_rows
    use_bf16 = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quant, torch_dtype=dtype,
        trust_remote_code=True, device_map={'':torch.cuda.current_device()}
    )
    if adapter:
        model = PeftModel.from_pretrained(model, str(adapter))
    model.eval()
    tokenizer.padding_side = 'left'
    stop_ids = [tokenizer.eos_token_id]
    end_token = tokenizer.convert_tokens_to_ids('<|end|>')
    if end_token is not None and end_token >= 0 and end_token not in stop_ids:
        stop_ids.append(end_token)
    records = []
    try:
        for i, record in enumerate(chosen, 1):
            encoded = tokenizer(make_prompt(record, architecture), return_tensors='pt',
                                add_special_tokens=False).to('cuda')
            output = model.generate(**encoded, max_new_tokens=max_new_tokens,
                                    do_sample=False, eos_token_id=stop_ids,
                                    pad_token_id=tokenizer.pad_token_id)
            raw = tokenizer.decode(output[0, encoded['input_ids'].shape[1]:],
                                   skip_special_tokens=True)
            obj, json_valid, pair_allowed = parse_answer(raw)
            records.append({
                'id':record['original_row_id'], 'architecture': architecture,
                'gold_category':record['crime_category'],
                'gold_subcategory':record['crime_subcategory'],
                'pred_category':obj.get('crime_category') if obj else None,
                'pred_subcategory':obj.get('crime_subcategory') if obj else None,
                'json_valid':json_valid, 'pair_allowed':pair_allowed,
                'raw_output':raw
            })
            if i%25==0 or i==len(chosen):
                print(f'{name}: {i}/{len(chosen)}')
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    dest = RESULTS/(name + '.jsonl')
    with dest.open('w',encoding='utf-8') as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False)+'\n')
    print('Predictions saved:', dest)
    return dest


## Smoke predictions


In [ ]:
predict_arch('C', name='C_base_smoke', limit=30)


In [ ]:
predict_arch('C', adapter=OUTPUTS/'C_smoke', name='C_ft_smoke', limit=30)


In [ ]:
predict_arch('A', name='A_base', limit=0)


In [ ]:
predict_arch('C', name='C_base', limit=0)


## Full test predictions


In [ ]:
predict_arch('C', adapter=OUTPUTS/'C', name='C_ft', limit=0)


## Evaluation metrics


In [ ]:
def macro_f1(y_true, y_pred):
    labels = set(y_true)
    f1s=[]
    for label in labels:
        tp=sum(a==label and b==label for a,b in zip(y_true,y_pred))
        fp=sum(a!=label and b==label for a,b in zip(y_true,y_pred))
        fn=sum(a==label and b!=label for a,b in zip(y_true,y_pred))
        f1s.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
    return sum(f1s)/len(f1s) if f1s else 0

def evaluate(name):
    path = RESULTS/(name+'.jsonl')
    predictions = read_jsonl(path)
    if not predictions:
        raise ValueError(f'{path} is empty')
    gt_c=[r['gold_category'] for r in predictions]
    gt_s=[r['gold_subcategory'] for r in predictions]
    pd_c=[r['pred_category'] if r['json_valid'] and isinstance(r['pred_category'],str) else '__INVALID__' for r in predictions]
    pd_s=[r['pred_subcategory'] if r['json_valid'] and isinstance(r['pred_subcategory'],str) else '__INVALID__' for r in predictions]
    n=len(predictions)
    metrics={
      'n':n,
      'category_accuracy':sum(a==b for a,b in zip(gt_c,pd_c))/n,
      'category_macro_f1':macro_f1(gt_c,pd_c),
      'subcategory_accuracy':sum(a==b for a,b in zip(gt_s,pd_s))/n,
      'subcategory_macro_f1':macro_f1(gt_s,pd_s),
      'pair_accuracy':sum(r['pair_allowed'] and a==c and b==d for r,a,b,c,d in zip(predictions,gt_c,gt_s,pd_c,pd_s))/n,
      'json_valid_rate':sum(bool(r['json_valid']) for r in predictions)/n,
      'allowed_pair_rate':sum(bool(r['pair_allowed']) for r in predictions)/n
    }
    (RESULTS/(name+'_metrics.json')).write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    print(name, json.dumps(metrics,indent=2))
    return metrics


In [ ]:
evaluate('A_base')
evaluate('A_ft')
evaluate('C_base')
evaluate('C_ft')


In [ ]:
def compare():
    names=['A_base','A_ft','C_base','C_ft']
    for name in names:
        if not (RESULTS/(name+'.jsonl')).exists():
            raise FileNotFoundError(f'Missing prediction file {name}.jsonl')
    gold_orders=[]
    for name in names:
        gold_orders.append([(r['id'],r['gold_category'],r['gold_subcategory'])
                            for r in read_jsonl(RESULTS/(name+'.jsonl'))])
    assert all(x==gold_orders[0] for x in gold_orders), 'Experiments use different test sets!'
    table=[]
    for name in names:
        p=RESULTS/(name+'_metrics.json')
        metrics=json.loads(p.read_text(encoding='utf-8')) if p.exists() else evaluate(name)
        row={'experiment':name,**metrics}
        table.append(row)
    csv_path=RESULTS/'comparison_A_C.csv'
    with csv_path.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=table[0].keys()); writer.writeheader();writer.writerows(table)
    for row in table:
        print(f"{row['experiment']:7}  Category Macro-F1={row['category_macro_f1']:.4f}  "
              f"Subcategory Macro-F1={row['subcategory_macro_f1']:.4f}  Pair acc={row['pair_accuracy']:.4f}")
    print('Comparison saved:',csv_path)


compare()


In [ ]:
import json
import math
from pathlib import Path

def load_predictions(name):
    path = RESULTS / f"{name}.jsonl"
    with path.open(encoding="utf-8") as f:
        return {
            str(r["id"]): r
            for line in f
            if (r := json.loads(line))
        }

A = load_predictions("A_ft")
C = load_predictions("C_ft")

assert set(A) == set(C), "Different test sets!"

def is_correct(r):
    return (
        r["gold_category"] == r["pred_category"]
        and r["gold_subcategory"] == r["pred_subcategory"]
        and r["pair_allowed"]
    )

both_correct = []
only_A = []
only_C = []
both_wrong = []

for event_id in A:
    a_ok = is_correct(A[event_id])
    c_ok = is_correct(C[event_id])

    if a_ok and c_ok:
        both_correct.append(event_id)
    elif a_ok:
        only_A.append(event_id)
    elif c_ok:
        only_C.append(event_id)
    else:
        both_wrong.append(event_id)

print("Both correct:", len(both_correct))
print("Only A correct:", len(only_A))
print("Only C correct:", len(only_C))
print("Both wrong:", len(both_wrong))


b = len(only_A)
c = len(only_C)
n = b + c

p_value = (
    min(1.0, 2 * sum(math.comb(n, k)
                    for k in range(min(b, c) + 1)) / 2**n)
    if n else 1.0
)

print("McNemar p-value:", p_value)


## Experiment limits


In [ ]:
def normalize_event(s):
    return " ".join(s.casefold().split())

train_events = {
    normalize_event(r["event"]) for r in train_rows
}

val_events = {
    normalize_event(r["event"]) for r in val_rows
}

test_events = {
    normalize_event(r["event"]) for r in test_rows
}

print("Train ∩ Validation:",
      len(train_events & val_events))

print("Train ∩ Test:",
      len(train_events & test_events))

print("Validation ∩ Test:",
      len(val_events & test_events))
